# Hierarchical Token Tree Visualization

This notebook visualizes the full hierarchical Chinese web-token tree release. It streams the gzip-compressed tree file, preserves the Featured 20 from the anonymous-review sample, and loads only requested records into memory.


In [1]:
import gzip
import html
import json
from pathlib import Path
from IPython.display import HTML, display

TREES_PATH = Path("hierarchical_chinese_web_token_trees.jsonl.gz")

if not TREES_PATH.exists():
    raise FileNotFoundError(f"Place {TREES_PATH.name} in the same directory as this notebook.")

FEATURED_CASES = [{'featured_token': '菲律宾申博', 'tree_id': 46049},
 {'featured_token': '北京赛车', 'tree_id': 62372},
 {'featured_token': '浪小辉', 'tree_id': 34617},
 {'featured_token': '大道香蕉', 'tree_id': 66136},
 {'featured_token': '大香蕉', 'tree_id': 18362},
 {'featured_token': '青青草', 'tree_id': 52527},
 {'featured_token': '就是博', 'tree_id': 21461},
 {'featured_token': '尼斯人', 'tree_id': 21578},
 {'featured_token': '巴体育', 'tree_id': 22211},
 {'featured_token': '娱乐官网', 'tree_id': 19520},
 {'featured_token': '冻传媒', 'tree_id': 9232},
 {'featured_token': '蜜桃传媒', 'tree_id': 82927},
 {'featured_token': '星空传媒', 'tree_id': 73481},
 {'featured_token': '老司机', 'tree_id': 44153},
 {'featured_token': ' 爱情岛', 'tree_id': 35863},
 {'featured_token': '大地资源', 'tree_id': 65982},
 {'featured_token': '剧在线', 'tree_id': 10269},
 {'featured_token': '传奇私服', 'tree_id': 59161},
 {'featured_token': '游戏盒', 'tree_id': 35037},
 {'featured_token': '拷锟斤拷', 'tree_id': 51584}]
_TREE_CACHE = {}

len(FEATURED_CASES), TREES_PATH


(20, WindowsPath('hierarchical_chinese_web_token_trees.jsonl.gz'))

## Collection Index

The default table shows the Featured 20 mapped to their current full-release records. Use `target_label`, `offset`, and `limit` to filter this bounded view.


In [2]:
def esc(value):
    return html.escape(str(value), quote=True)


def visible_token(value):
    return str(value).replace(" ", "[space]")


def iter_jsonl_gz(path):
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                yield json.loads(line)


def iter_nodes(root):
    stack = [root]
    while stack:
        node = stack.pop()
        yield node
        stack.extend(reversed(node.get("children", [])))


def find_node(root, token):
    for node in iter_nodes(root):
        if node.get("token") == token:
            return node
    return None


def count_distinct_tokens(root):
    return len({node.get("token") for node in iter_nodes(root)})


def tree_max_depth(root, depth_offset=0):
    return max(node.get("depth", 0) - depth_offset for node in iter_nodes(root))


FEATURED_BY_TREE_ID = {case["tree_id"]: case for case in FEATURED_CASES}
FEATURED_BY_TOKEN = {case["featured_token"].strip(): case for case in FEATURED_CASES}


def cache_tree(record):
    _TREE_CACHE[record["tree_id"]] = record
    _TREE_CACHE[record["collection_id"]] = record
    _TREE_CACHE[record["root"]["token"].strip()] = record
    return record


def load_featured_records():
    missing = set(FEATURED_BY_TREE_ID) - {
        key for key in _TREE_CACHE if isinstance(key, int)
    }
    if missing:
        for record in iter_jsonl_gz(TREES_PATH):
            tree_id = record.get("tree_id")
            if tree_id in missing:
                cache_tree(record)
                missing.remove(tree_id)
                if not missing:
                    break
    if missing:
        raise KeyError(f"Featured tree IDs not found: {sorted(missing)}")
    return [_TREE_CACHE[case["tree_id"]] for case in FEATURED_CASES]


INDEX_CSS = """
<style>
.index-wrap {font-family:Inter,Segoe UI,sans-serif; color:#17202a; max-width:1120px}
.index-note {color:#5d6d7e; margin:0 0 10px 0; font-size:13px}
.index-table {border-collapse:collapse; width:100%; font-size:13px; background:#fff}
.index-table th {background:#263746; color:#fff; text-align:left; padding:9px; border:1px solid #d5dde3}
.index-table td {padding:8px 9px; border:1px solid #d5dde3; vertical-align:top}
.index-table tr:nth-child(even) {background:#f4f7f8}
.index-table tr:hover {background:#eaf3f5}
.index-table code {background:#eef2f3; color:#17202a; padding:2px 4px; border-radius:3px}
</style>
"""


def collection_index_html(limit=20, offset=0, target_label=None):
    records = load_featured_records()
    rows = []
    for case, record in zip(FEATURED_CASES, records):
        if target_label and record["target_label"] != target_label:
            continue
        source_tree = record["hierarchical_tree"]
        view_tree = find_node(source_tree, case["featured_token"])
        if view_tree is None:
            raise KeyError(
                f"Featured token {case['featured_token']!r} is missing from tree {record['tree_id']}"
            )
        depth_offset = view_tree.get("depth", 0)
        shown_root = view_tree.get("token", "")
        shown_label = view_tree.get("label", "")
        token_count = count_distinct_tokens(view_tree)
        rows.append(
            "<tr>"
            f"<td>{esc(record['collection_id'])}</td>"
            f"<td>{esc(record['tree_id'])}</td>"
            f"<td><code>{esc(visible_token(shown_root))}</code></td>"
            f"<td>{esc(shown_label)}</td>"
            f"<td>{esc(record['target_label'])}</td>"
            f"<td>{esc(token_count)}</td>"
            f"<td>{esc(tree_max_depth(view_tree, depth_offset))}</td>"
            "</tr>"
        )
    selected = rows[offset : offset + limit if limit is not None else None]
    table = """
    <div class="index-wrap">
      <div class="index-note">Featured 20 mapped from the anonymous-review sample to the full release.</div>
      <table class="index-table">
        <thead><tr>
          <th>collection_id</th><th>tree_id</th><th>root token</th>
          <th>root label</th><th>target label</th><th>tokens</th><th>max depth</th>
        </tr></thead>
        <tbody>{rows}</tbody>
      </table>
    </div>
    """.format(rows="".join(selected))
    return INDEX_CSS + table


def show_collection_index(limit=20, offset=0, target_label=None):
    display(HTML(collection_index_html(limit, offset, target_label)))


show_collection_index()


collection_id,tree_id,root token,root label,target label,tokens,max depth
full_release_046049,46049,菲律宾申博,Online Gambling,Online Gambling,353,7
full_release_062372,62372,北京赛车,Online Gambling,Online Gambling,39,5
full_release_034617,34617,浪小辉,Adult Content,Adult Content,13,3
full_release_066136,66136,大道香蕉,Adult Content,Adult Content,56,5
full_release_018362,18362,大香蕉,Anomalous,Adult Content,304,7
full_release_052527,52527,青青草,Adult Content,Adult Content,433,7
full_release_021461,21461,就是博,Online Gambling,Online Gambling,52,5
full_release_021578,21578,尼斯人,Online Gambling,Online Gambling,43,4
full_release_022211,22211,巴体育,Online Gambling,Online Gambling,7,2
full_release_019520,19520,娱乐官网,Online Gambling,Online Gambling,83,3


## Tree Viewer

`display_tree` accepts a current `tree_id`, `collection_id`, or token from the Featured 20.


In [3]:
TREE_CSS = """
<style>
.tree-wrap {font-family:Inter,Segoe UI,sans-serif; color:#17202a; max-width:1180px}
.tree-title {font-size:22px; font-weight:700; margin:4px 0 6px}
.tree-title code,.tree-wrap .token {background:#e9f0f2; color:#17202a; padding:2px 5px; border-radius:3px}
.tree-meta {color:#60717d; font-size:12px; margin-bottom:10px}
.reason {background:#e8f3ee; border-left:4px solid #2f8065; padding:10px 12px; margin:9px 0; line-height:1.45}
.composition {background:#fff5cf; border-left:4px solid #d3a62e; padding:10px 12px; margin:9px 0; line-height:1.45}
.node,.leaf {margin-left:18px; border-left:1px solid #b7c4cc; padding:4px 0 4px 12px}
.node summary {cursor:pointer; list-style:revert}
.children {margin-left:4px}
.label {color:#0969a8; margin-left:7px; font-weight:600}
.meta,.exclude {color:#6f7f89; margin-left:7px; font-size:11px}
.not-covered {opacity:.62}
.context {white-space:pre-wrap; overflow-wrap:anywhere; background:#f4f7f8; border:1px solid #d5dde3; padding:12px; max-height:520px; overflow:auto}
</style>
"""


def resolve_tree_record(key):
    normalized = key.strip() if isinstance(key, str) else key
    if normalized in _TREE_CACHE:
        return _TREE_CACHE[normalized]
    case = FEATURED_BY_TOKEN.get(normalized) if isinstance(normalized, str) else None
    if case:
        key = case["tree_id"]
        if key in _TREE_CACHE:
            return _TREE_CACHE[key]
    target_tree_id = key if isinstance(key, int) else None
    if isinstance(key, str) and key.startswith("full_release_") and key[13:].isdigit():
        target_tree_id = int(key[13:])
    for record in iter_jsonl_gz(TREES_PATH):
        if target_tree_id is not None and record.get("tree_id") == target_tree_id:
            return cache_tree(record)
        if target_tree_id is None and record["root"]["token"].strip() == normalized:
            return cache_tree(record)
    raise KeyError(f"No tree record found for {key!r}")


def featured_view(record, key):
    normalized = key.strip() if isinstance(key, str) else key
    case = FEATURED_BY_TOKEN.get(normalized) if isinstance(normalized, str) else None
    if case is None:
        case = FEATURED_BY_TREE_ID.get(record["tree_id"])
    tree = record["hierarchical_tree"]
    if case:
        node = find_node(tree, case["featured_token"])
        if node is None:
            raise KeyError(f"Featured token {case['featured_token']!r} is missing")
        return node, node.get("depth", 0), case
    return tree, 0, case


def node_html(node, max_show_depth=4, depth_offset=0):
    children = node.get("children", [])
    depth = node.get("depth", 0) - depth_offset
    token = esc(visible_token(node.get("token", "")))
    label = esc(node.get("label", ""))
    covered = node.get("reason_covered", True)
    coverage_class = "" if covered else " not-covered"
    excluded = ""
    if node.get("exclude_reason"):
        excluded = f'<span class="exclude">{esc(node["exclude_reason"])}</span>'
    summary = (
        f'<span class="token">{token}</span>'
        f'<span class="label">{label}</span>'
        f'<span class="meta">depth={esc(depth)}, children={esc(len(children))}</span>'
        f'{excluded}'
    )
    can_expand = children and (max_show_depth is None or depth < max_show_depth)
    if can_expand:
        rendered = "".join(node_html(child, max_show_depth, depth_offset) for child in children)
        return f'<details open class="node{coverage_class}"><summary>{summary}</summary><div class="children">{rendered}</div></details>'
    if children:
        summary += '<span class="meta">collapsed below max_depth</span>'
    return f'<div class="leaf{coverage_class}">{summary}</div>'


def composition_html(record):
    composition = record.get("composition")
    if not composition:
        return ""
    parts = []
    for item in composition.get("composition_tokens", []):
        parts.append(
            f'<span class="token">{esc(visible_token(item["token"]))}</span>'
            f'<span class="label">{esc(item["label"])}</span>'
        )
    left = " + ".join(parts)
    right = f'<span class="token">{esc(visible_token(composition["composite_token"]))}</span>'
    return (
        f'<div class="composition"><b>Composition:</b> {left} &rarr; {right}<br>'
        f'{esc(composition.get("interpretation", ""))}</div>'
    )


def reason_html(record):
    reason = record.get("classification_reason")
    text = reason if reason else "Unavailable because suitable web evidence was not found."
    return f'<div class="reason"><b>Classification reason:</b> {esc(text)}</div>'


def display_tree(key, max_depth=4):
    record = resolve_tree_record(key)
    tree, depth_offset, _ = featured_view(record, key)
    header = f"""
    <div class="tree-wrap">
      <div class="tree-title"><code>{esc(visible_token(tree['token']))}</code></div>
      <div class="tree-meta">
        collection_id={esc(record['collection_id'])}; tree_id={esc(record['tree_id'])};
        root_label={esc(tree['label'])}; target_label={esc(record['target_label'])};
        tokens={esc(count_distinct_tokens(tree))}; max_depth={esc(tree_max_depth(tree, depth_offset))};
        representative_token=<code>{esc(visible_token(record['representative_token']))}</code>
      </div>
      {reason_html(record)}
      {composition_html(record)}
      {node_html(tree, max_depth, depth_offset)}
    </div>
    """
    display(HTML(TREE_CSS + header))


## Examples


In [4]:
display_tree(34617, max_depth=4)


In [5]:
display_tree("菲律宾申博", max_depth=3)


In [6]:
display_tree(62372, max_depth=4)
